# Demo for Pydantic-output-structuring and embedding-vectors using Mistral API

- Buy tokens: https://admin.mistral.ai/organization/billing
- Configure api-key https://console.mistral.ai/api-keys/
- Select Model https://docs.mistral.ai/getting-started/models/models_overview/
- Run all cells [Runtime tab -> run all]
- Type text when prompted at bottom of the colab. 'exit' to stop. You can change models mid-conversation.
- Download the saved conversation history if you want. (optional)


See: https://pypi.org/project/mistralai/

In [ ]:
# Install the Mistral SDK and Pydantic (if you don't have it)

try:  # Check if you have it
    from mistralai.client import Mistral

except ImportError:  # (if you don't have it)
    !pip install -q --upgrade mistralai pydantic


In [ ]:
"""
Google Colab demo: Mistral API connection with
  (1) Pydantic structured output (biology fields as JSON), and
  (2) Mistral vector embeddings (mistral-embed) with cosine similarity.

Requires: mistralai (client import path: mistralai.client), pydantic >= 2.
Python >= 3.10 (required by the mistralai SDK).

API key lookup order (the key value is never written in this file):
  1. Colab Secrets (left sidebar, key icon), using the secret name set below.
  2. An operating-system environment variable with the same name.
  3. An interactive hidden prompt (getpass).
"""

import getpass
import json
import math
import os
import traceback

from pydantic import BaseModel, ConfigDict, Field, ValidationError

from mistralai.client import Mistral, errors as mistral_errors


# ---------------------------------------------------------------------------
# Configuration: names of settings, not secret values.
# ---------------------------------------------------------------------------
MISTRAL_API_KEY_SECRET_NAME: str = "mistral_api_key"  # name of the Colab secret / environment variable
MISTRAL_CHAT_MODEL_NAME: str = "mistral-small-latest"  # chat model (name taken from the SDK README)
MISTRAL_EMBEDDING_MODEL_NAME: str = "mistral-embed"  # embedding model (name taken from the SDK README)
STRUCTURED_OUTPUT_SCHEMA_NAME: str = "organism_profile"  # label sent to the API for the JSON schema


# ---------------------------------------------------------------------------
# Pydantic schema: the structure the model output must follow.
# extra="forbid" makes Pydantic emit "additionalProperties": false in the JSON
# schema, which strict schema enforcement on the API side expects.
# ---------------------------------------------------------------------------
class Taxonomy(BaseModel):
    """Linnaean taxonomic classification of one organism."""

    model_config = ConfigDict(extra="forbid")

    domain: str = Field(description="Domain, e.g. Eukaryota, Bacteria, Archaea")
    kingdom: str = Field(description="Kingdom, e.g. Animalia, Plantae, Fungi")
    phylum: str = Field(description="Phylum, e.g. Chordata")
    taxonomic_class: str = Field(description="Biological class, e.g. Amphibia, Mammalia")
    genus: str = Field(description="Genus name, e.g. Ambystoma")
    species: str = Field(description="Species epithet, e.g. mexicanum")


class BiologicalAdaptation(BaseModel):
    """One specialised trait of an organism and the function it serves."""

    model_config = ConfigDict(extra="forbid")

    trait_name: str = Field(description="Name of the specialised trait or organelle")
    function: str = Field(description="How this trait aids survival or biological function")


class OrganismProfile(BaseModel):
    """Structured biological profile of a single organism."""

    model_config = ConfigDict(extra="forbid")

    common_name: str = Field(description="Common name, e.g. Axolotl")
    scientific_name: str = Field(description="Binomial name, e.g. Ambystoma mexicanum")
    cellular_organization: str = Field(
        description="e.g. Unicellular Prokaryote, Multicellular Eukaryote"
    )
    trophic_level: str = Field(
        description="e.g. Autotroph, Carnivorous Heterotroph, Decomposer"
    )
    taxonomy: Taxonomy
    adaptations: list[BiologicalAdaptation]
    # Required but nullable: the key must be present, and its value may be null
    # when the text does not state a conservation status.
    conservation_status: str | None = Field(
        description="IUCN status such as Critically Endangered or Least Concern; null if the text does not say"
    )


# ---------------------------------------------------------------------------
# API key retrieval
# ---------------------------------------------------------------------------
def retrieve_mistral_api_key(secret_name: str) -> str:
    """
    Retrieve the Mistral API key from Colab Secrets, then the OS environment,
    then an interactive hidden prompt.

    Args:
        secret_name: Name of the Colab secret and of the environment variable
            that hold the API key (the name only, never the key itself).

    Returns:
        The API key as a non-empty string.

    Raises:
        ValueError: If no non-empty key could be obtained from any source.
    """
    retrieved_api_key: str | None = None

    # Source 1: Colab Secrets. The import only succeeds inside Google Colab.
    try:
        from google.colab import userdata  # type: ignore[import-not-found]

        retrieved_api_key = userdata.get(secret_name)
    except ImportError:
        print("Not running inside Google Colab, so Colab Secrets are skipped.")
    except Exception as colab_secret_error:
        # Typical causes: secret missing, or notebook access toggle is off.
        print(
            f"Colab Secret '{secret_name}' could not be read "
            f"({type(colab_secret_error).__name__}: {colab_secret_error}). "
            "Trying the next source."
        )

    # Source 2: operating-system environment variable.
    if not retrieved_api_key:
        retrieved_api_key = os.environ.get(secret_name)

    # Source 3: interactive hidden prompt.
    if not retrieved_api_key:
        retrieved_api_key = getpass.getpass(f"Enter value for {secret_name}: ")

    if not retrieved_api_key or not retrieved_api_key.strip():
        raise ValueError(
            f"No API key was provided. Add a Colab Secret named '{secret_name}' "
            "(and enable notebook access), set an environment variable with that "
            "name, or type the key at the prompt."
        )
    return retrieved_api_key.strip()


# ---------------------------------------------------------------------------
# Part 1: structured output
# ---------------------------------------------------------------------------
def build_json_schema_response_format(
    pydantic_model_class: type[BaseModel],
    schema_name: str,
) -> dict:
    """
    Convert a Pydantic model class into the `response_format` dictionary that
    the Mistral chat completion endpoint accepts for JSON-schema output.

    Args:
        pydantic_model_class: The Pydantic model that defines the output structure.
        schema_name: Label for the schema, sent to the API.

    Returns:
        A dictionary of the form
        {"type": "json_schema", "json_schema": {"name": ..., "schema": ..., "strict": True}}.
    """
    json_schema_dictionary: dict = pydantic_model_class.model_json_schema()
    return {
        "type": "json_schema",
        "json_schema": {
            "name": schema_name,
            "schema": json_schema_dictionary,
            "strict": True,
        },
    }


def extract_organism_profile_from_text(
    mistral_client: Mistral,
    source_biology_text: str,
    chat_model_name: str,
) -> OrganismProfile:
    """
    Ask a Mistral chat model to extract an OrganismProfile from free text, then
    validate the reply with Pydantic.

    Args:
        mistral_client: An open Mistral SDK client.
        source_biology_text: Free-text biological description to extract from.
        chat_model_name: Name of the Mistral chat model.

    Returns:
        A validated OrganismProfile instance.

    Raises:
        RuntimeError: If the API reply has no usable text content.
        ValidationError: If the reply is not valid JSON for OrganismProfile.
        mistral_errors.MistralError: If the API call itself fails (HTTP error).
    """
    chat_response = mistral_client.chat.complete(
        model=chat_model_name,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a bioinformatics assistant. Extract structured "
                    "biological data from the text. Use only information in the "
                    "text. Use null where the text does not state a value that "
                    "is allowed to be null."
                ),
            },
            {"role": "user", "content": source_biology_text},
        ],
        response_format=build_json_schema_response_format(
            OrganismProfile, STRUCTURED_OUTPUT_SCHEMA_NAME
        ),
        temperature=0.0,
        stream=False,
    )

    if not chat_response.choices:
        raise RuntimeError("The chat response contained no choices.")

    reply_content = chat_response.choices[0].message.content
    if not isinstance(reply_content, str) or not reply_content.strip():
        raise RuntimeError(
            "The chat reply content is not a non-empty string "
            f"(received type: {type(reply_content).__name__}). "
            "Print the full response object to inspect its structure."
        )

    # Pydantic parses the JSON text and checks every field and type.
    return OrganismProfile.model_validate_json(reply_content)


# ---------------------------------------------------------------------------
# Part 2: embeddings
# ---------------------------------------------------------------------------
def create_text_embeddings(
    mistral_client: Mistral,
    input_text_list: list[str],
    embedding_model_name: str,
) -> list[list[float]]:
    """
    Create one embedding vector per input text with the Mistral embeddings API.

    Args:
        mistral_client: An open Mistral SDK client.
        input_text_list: Texts to embed (must contain at least one string).
        embedding_model_name: Name of the Mistral embedding model.

    Returns:
        A list of embedding vectors, in the same order as input_text_list.

    Raises:
        ValueError: If input_text_list is empty.
        RuntimeError: If the number of returned vectors differs from the number of inputs.
        mistral_errors.MistralError: If the API call itself fails (HTTP error).
    """
    if not input_text_list:
        raise ValueError("input_text_list is empty; provide at least one string to embed.")

    embedding_response = mistral_client.embeddings.create(
        model=embedding_model_name,
        inputs=input_text_list,
    )

    # Sort by the index field so output order is guaranteed to match input order.
    sorted_embedding_items = sorted(embedding_response.data, key=lambda item: item.index)
    embedding_vector_list = [list(item.embedding) for item in sorted_embedding_items]

    if len(embedding_vector_list) != len(input_text_list):
        raise RuntimeError(
            f"Sent {len(input_text_list)} texts but received "
            f"{len(embedding_vector_list)} embedding vectors."
        )
    return embedding_vector_list


def compute_cosine_similarity(
    first_vector: list[float],
    second_vector: list[float],
) -> float:
    """
    Compute the cosine similarity between two equal-length vectors.

    Args:
        first_vector: First vector.
        second_vector: Second vector.

    Returns:
        A float in [-1.0, 1.0]; higher means more similar direction.

    Raises:
        ValueError: If the lengths differ or either vector has zero magnitude.
    """
    if len(first_vector) != len(second_vector):
        raise ValueError(
            f"Vector lengths differ: {len(first_vector)} vs {len(second_vector)}."
        )
    dot_product = sum(a * b for a, b in zip(first_vector, second_vector))
    first_magnitude = math.sqrt(sum(a * a for a in first_vector))
    second_magnitude = math.sqrt(sum(b * b for b in second_vector))
    if first_magnitude == 0.0 or second_magnitude == 0.0:
        raise ValueError("Cosine similarity is undefined for a zero-magnitude vector.")
    return dot_product / (first_magnitude * second_magnitude)


# ---------------------------------------------------------------------------
# Demo entry point
# ---------------------------------------------------------------------------
SAMPLE_BIOLOGY_TEXT: str = """
The Axolotl (Ambystoma mexicanum) is a paedomorphic salamander closely related to the tiger salamander.
Belonging to domain Eukaryota, kingdom Animalia, phylum Chordata, class Amphibia, and genus Ambystoma,
it is a multicellular eukaryote native exclusively to Lake Xochimilco in Mexico.

Unlike most amphibians, axolotls exhibit neoteny: they reach adulthood without undergoing metamorphosis,
retaining external feathery gills along their necks that allow them to absorb dissolved oxygen directly from water.
They also possess remarkable tissue regeneration, able to regrow complete limbs, damaged heart tissue, and sections
of the central nervous system without scarring. They are carnivorous heterotrophs that consume worms, mollusks,
and small fish by suction feeding. Unfortunately, habitat degradation has made them Critically Endangered.
"""

EMBEDDING_DEMO_TEXT_LIST: list[str] = [
    "Axolotls regenerate lost limbs and keep external gills throughout adult life.",
    "Salamanders can regrow tails and limbs after injury.",
    "Mitochondria produce ATP through oxidative phosphorylation.",
]


def run_mistral_demo() -> None:
    """
    Run both demos in sequence: structured extraction, then embeddings.
    Every failure is announced with a specific message and a full traceback.
    """
    try:
        mistral_api_key = retrieve_mistral_api_key(MISTRAL_API_KEY_SECRET_NAME)

        # The context manager closes the SDK's HTTP connections on exit.
        with Mistral(api_key=mistral_api_key) as mistral_client:

            # ---- Part 1: structured output ----
            print("Part 1: sending structured-output request to Mistral...")
            organism_profile = extract_organism_profile_from_text(
                mistral_client=mistral_client,
                source_biology_text=SAMPLE_BIOLOGY_TEXT,
                chat_model_name=MISTRAL_CHAT_MODEL_NAME,
            )
            print("\n--- Pydantic object ---")
            print(repr(organism_profile))
            print("\n--- Validated JSON ---")
            print(organism_profile.model_dump_json(indent=2))

            # ---- Part 2: embeddings ----
            print("\nPart 2: sending embeddings request to Mistral...")
            embedding_vector_list = create_text_embeddings(
                mistral_client=mistral_client,
                input_text_list=EMBEDDING_DEMO_TEXT_LIST,
                embedding_model_name=MISTRAL_EMBEDDING_MODEL_NAME,
            )
            print(f"Number of vectors returned: {len(embedding_vector_list)}")
            print(f"Dimensions per vector: {len(embedding_vector_list[0])}")
            print(f"First 5 values of vector 0: {embedding_vector_list[0][:5]}")

            for first_index, second_index in [(0, 1), (0, 2)]:
                similarity_score = compute_cosine_similarity(
                    embedding_vector_list[first_index],
                    embedding_vector_list[second_index],
                )
                print(
                    f"Cosine similarity, text {first_index} vs text {second_index}: "
                    f"{similarity_score:.4f}"
                )

    except mistral_errors.MistralError as mistral_http_error:
        print("Mistral API returned an HTTP error:")
        print(f"  status code: {mistral_http_error.status_code}")
        print(f"  message:     {mistral_http_error.message}")
        print(f"  body:        {mistral_http_error.body}")
        traceback.print_exc()
    except ValidationError as pydantic_validation_error:
        print(
            "The model reply did not match the OrganismProfile schema. "
            "Details follow:"
        )
        print(pydantic_validation_error)
        traceback.print_exc()
    except (ValueError, RuntimeError) as demo_input_or_response_error:
        print(f"Demo stopped: {demo_input_or_response_error}")
        traceback.print_exc()
    except Exception as unexpected_error:
        print(
            f"Unexpected {type(unexpected_error).__name__}: {unexpected_error}. "
            "If this is an ImportError, run the install cell and restart the session."
        )
        traceback.print_exc()


run_mistral_demo()

Part 1: sending structured-output request to Mistral...

--- Pydantic object ---
OrganismProfile(common_name='Axolotl', scientific_name='Ambystoma mexicanum', cellular_organization='Multicellular Eukaryote', trophic_level='Carnivorous Heterotroph', taxonomy=Taxonomy(domain='Eukaryota', kingdom='Animalia', phylum='Chordata', taxonomic_class='Amphibia', genus='Ambystoma', species='mexicanum'), adaptations=[BiologicalAdaptation(trait_name='Neoteny', function='Reaches adulthood without undergoing metamorphosis, retaining external feathery gills for direct oxygen absorption from water'), BiologicalAdaptation(trait_name='External feathery gills', function='Allows direct absorption of dissolved oxygen from water'), BiologicalAdaptation(trait_name='Tissue regeneration', function='Able to regrow complete limbs, damaged heart tissue, and sections of the central nervous system without scarring'), BiologicalAdaptation(trait_name='Suction feeding', function='Consumes worms, mollusks, and small fish